In [ ]:
#!/usr/bin/env python3
"""Docking-score ROC analysis for the ensemble-based virtual screen (EBVS).

Ligands are ranked by the Boltzmann-weighted average of their Glide docking
scores across the 50-receptor ensemble; the ROC curve is measured against the
14 known binders. Writes one figure: figure_6_EBVS_<method>_roc.png
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from pathlib import Path
import re

IN_CSV  = Path("data/combined_receptors_scores.csv")
OUT_PNG = Path("figure_6_EBVS_T-REMD_roc.png")
N_HITS  = 14            # leading rows of IN_CSV are the known binders

FIG_W_SQ = 3.6
FIG_H_SQ = 3.6

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 1.5,
    "lines.linewidth": 2,
    "lines.markersize": 4.5,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02
})

R = 0.0019872041       # kcal/mol/K
T_DEFAULT = 298.15

# numpy >= 2.0 renamed trapz to trapezoid
_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


def avg_boltz(E, T=T_DEFAULT):
    """Boltzmann-weighted average of energies across receptors for each ligand.

    p_i is proportional to exp(-(E_i - Emin)/(RT)), weights normalized per ligand.
    """
    all_nan = np.isnan(E).all(axis=1)
    Emin = np.nanmin(E, axis=1, keepdims=True)
    Emin[np.isnan(Emin)] = 0.0  # safe fill for rows that were all NaN
    w = np.exp(-(E - Emin) / (R * T))
    num = np.nansum(E * w, axis=1)
    den = np.nansum(w, axis=1)
    out = num / den
    out[all_nan] = np.nan
    return out


def roc_points_and_auc(y_true, scores):
    """Manual ROC + trapezoid AUC. Higher 'scores' indicate more likely HIT."""
    mask = ~np.isnan(scores)
    yv = y_true[mask].astype(int)
    sv = scores[mask].astype(float)

    if yv.size == 0 or yv.sum() == 0 or (yv == 0).sum() == 0:
        return 0.5, np.array([0.0, 1.0]), np.array([0.0, 1.0])

    order = np.argsort(-sv)
    yv = yv[order]
    sv = sv[order]

    P = yv.sum()
    Nn = yv.size - P
    tps = np.cumsum(yv)
    fps = np.cumsum(1 - yv)

    # keep only thresholds where the score actually changes
    diffs = np.diff(sv)
    idx_unique = np.r_[np.where(diffs != 0)[0], yv.size - 1]

    TPR = np.r_[0.0, tps[idx_unique] / P, 1.0]
    FPR = np.r_[0.0, fps[idx_unique] / Nn, 1.0]

    return float(_trapz(TPR, FPR)), FPR, TPR


df = pd.read_csv(IN_CSV)

# The leading N_HITS rows are the known binders; everything else is a decoy.
if not any(c.lower() == "hit" for c in df.columns):
    df.insert(0, "HIT", 0)
    df.loc[:N_HITS - 1, "HIT"] = 1

label_col = next(c for c in df.columns if c.lower() == "hit")
y = df[label_col].astype(int).clip(0, 1).to_numpy()

receptor_cols = sorted(
    [c for c in df.columns if str(c).lower().startswith("receptor_")],
    key=lambda c: int(re.search(r"(\d+)", str(c)).group(1))
)
E_full = df[receptor_cols].to_numpy(dtype=float)
KMAX = E_full.shape[1]

energy = avg_boltz(E_full)      # lower = better
scores = -energy                # higher = more likely HIT
auc_val, fpr, tpr = roc_points_and_auc(y, scores)

fig = plt.figure(figsize=(FIG_W_SQ, FIG_H_SQ))
ax = plt.gca()
ax.plot(fpr, tpr, linewidth=2, color="red", label=f"AUC={auc_val:.3f}, k={KMAX}")
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.0, color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(frameon=False, loc="lower right")
ax.minorticks_off()
ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

fig.savefig(OUT_PNG, dpi=600)
plt.close(fig)
print(f"AUC={auc_val:.3f}  k={KMAX}  hits={int(y.sum())}  ligands={len(y)}  ->  {OUT_PNG}")
